# 05 - Vector DB + Embeddings


## What this notebook is for

This notebook is written as **notes + code**. Read the Markdown cells first, then run the code cells.

The goal is to understand the concept, see the architecture, implement a small example, and then evaluate/improve it.

## Learning objectives

- Understand what an embedding index stores.
- Compare FAISS and Chroma conceptually.
- Store vectors, text, and metadata.
- Understand distance vs similarity.

## Core AI concept

A vector database/index stores embeddings so a query embedding can be compared with many document embeddings efficiently. In production, the vector alone is not enough: you need the original text, metadata, filtering, and a mapping from retrieved IDs back to source content.

## Architecture / mental model

```text
Text
 ↓ embedding model
Vector + metadata
 ↓
FAISS / Chroma / other vector store
 ↓
Top-k similar records
```

## Connection to the supplied learning material

Your `embeddings.ipynb`, `faiss.ipynb`, and `chroma_db.ipynb` cover embeddings, cosine similarity, FAISS indexes, persistence, and Chroma storage.

### Source foundation
This notebook is an expanded learning version of the uploaded notebooks. The source material for this track includes:
- `chunking.ipynb`
- `embeddings.ipynb`
- `faiss.ipynb`
- `chroma_db.ipynb`
- `searching.ipynb`
- `rag.ipynb`

The original examples and terminology are retained where useful, then organized into a cleaner learning sequence.

## Setup

The code below uses environment variables for secrets and model names so the notebook is easier to maintain when providers or model versions change.

In [ ]:
from pathlib import Path
import os
from dotenv import load_dotenv

load_dotenv()

OPENAI_MODEL = os.getenv("OPENAI_MODEL", "gpt-5.6-luna")
OPENAI_EMBED_MODEL = os.getenv("OPENAI_EMBED_MODEL", "text-embedding-3-small")

print("Model:", OPENAI_MODEL)
print("Embedding model:", OPENAI_EMBED_MODEL)

In [ ]:
%pip install -qU faiss-cpu chromadb sentence-transformers

### Embeddings + FAISS

In [ ]:
from sentence_transformers import SentenceTransformer
import numpy as np
import faiss

documents = [
    "Python is a programming language.",
    "RAG retrieves information before generation.",
    "Agents can use tools to perform actions.",
    "Vector databases support similarity search.",
]

embed_model = SentenceTransformer("all-MiniLM-L6-v2")
vectors = embed_model.encode(documents).astype("float32")

index = faiss.IndexFlatL2(vectors.shape[1])
index.add(vectors)

query = "How does RAG work?"
q = embed_model.encode([query]).astype("float32")

distances, indices = index.search(q, k=2)

for d, i in zip(distances[0], indices[0]):
    print("distance:", float(d))
    print("document:", documents[i])

### Chroma stores text + metadata

In [ ]:
import chromadb

client = chromadb.PersistentClient(path="./chroma_local")
collection = client.get_or_create_collection("demo")

collection.upsert(
    ids=["1", "2", "3"],
    documents=documents[:3],
    metadatas=[
        {"topic": "python"},
        {"topic": "rag"},
        {"topic": "agents"},
    ],
)

results = collection.query(
    query_texts=["How does RAG work?"],
    n_results=2,
)

print(results["documents"][0])
print(results["metadatas"][0])

## Evaluation

Evaluate retrieval by checking rank, distance/similarity, and expected source coverage. For production, benchmark latency and memory use too.

## Improvement ideas

Normalize vectors when appropriate, choose the metric deliberately, use metadata filters, tune top-k, consider approximate nearest-neighbor indexes for scale, and keep document IDs stable.

## Practice exercises

1. Compare L2 and cosine-style retrieval.
2. Add metadata filtering.
3. Save and reload a FAISS index.

## Final checklist

- [ ] I can explain the concept in simple words.
- [ ] I can describe where it fits in a GenAI system.
- [ ] I can run and modify the code.
- [ ] I know at least one way to evaluate it.
- [ ] I know what to change when it fails.

## Research / reference notes


- LangChain Python reference: https://reference.langchain.com/python/langchain/overview
- LangGraph Python reference: https://reference.langchain.com/python/langgraph/overview
- Ragas metrics: https://docs.ragas.io/en/latest/concepts/metrics/available_metrics/
- OpenAI function calling guide: https://developers.openai.com/api/docs/guides/function-calling
- OpenAI Responses API: https://developers.openai.com/api/reference/cli/resources/responses/methods/create